# SAM 3 (Segment Anything Model 3) Demo

This notebook demonstrates how to use Meta's SAM 3 for image segmentation.

**Requirements:**
- Python 3.12+
- PyTorch 2.7+
- CUDA 12.6+ (for GPU acceleration)

## 1. Installation

```bash
pip install transformers torch torchvision requests pillow matplotlib numpy
```

**Note:** You need to authenticate with HuggingFace to access the model weights.

In [ ]:
# SAM 3 requires transformers from main branch (not yet in stable release)
!pip install git+https://github.com/huggingface/transformers
!pip install torch torchvision requests pillow matplotlib numpy python-dotenv huggingface_hub

## 2. Hugging Face Authentication

SAM 3 model weights are **gated** and require access approval.

1. Go to https://huggingface.co/facebook/sam3 and click **"Request Access"**
2. Generate an access token at https://huggingface.co/settings/tokens
3. Run the cell below to authenticate

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# Load token from .env file
load_dotenv()
token = os.environ.get("HF_TOKEN")

# Login to HuggingFace
login(token=token)
print("Authenticated with HuggingFace!")

## 3. Download Sample Images

In [ ]:
import os
import requests
from pathlib import Path

# Create images directory
images_dir = Path("images")
images_dir.mkdir(exist_ok=True)

# Sample images from Unsplash (free to use)
sample_images = {
    "dog.jpg": "https://images.unsplash.com/photo-1587300003388-59208cc962cb?w=800",
    "city.jpg": "https://images.unsplash.com/photo-1480714378408-67cf0d13bc1b?w=800",
    "food.jpg": "https://images.unsplash.com/photo-1546069901-ba9599a7e63c?w=800",
    "car.jpg": "https://images.unsplash.com/photo-1494976388531-d1058494cdd8?w=800",
    "people.jpg": "https://images.unsplash.com/photo-1529156069898-49953e39b3ac?w=800",
}

def download_image(url, filepath):
    """Download an image from URL."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    with open(filepath, "wb") as f:
        f.write(response.content)
    print(f"Downloaded: {filepath}")

# Download all sample images
for filename, url in sample_images.items():
    filepath = images_dir / filename
    if not filepath.exists():
        try:
            download_image(url, filepath)
        except Exception as e:
            print(f"Failed to download {filename}: {e}")
    else:
        print(f"Already exists: {filepath}")

print(f"\nImages in {images_dir}:")
for img in images_dir.glob("*.jpg"):
    print(f"  - {img.name}")

## 4. Import SAM 3 and Setup Model

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests

# Check device (CUDA > MPS > CPU)
if torch.cuda.is_available():
    device = "cuda"
    print(f"Using device: {device}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = "mps"
    print(f"Using device: {device} (Apple Silicon)")
else:
    device = "cpu"
    print(f"Using device: {device}")

In [ ]:
# Import SAM 3 from Transformers (direct import path)
from transformers.models.sam3 import Sam3Processor, Sam3Model

# Load the model (downloads weights automatically if authenticated)
print("Loading SAM 3 model...")
model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")
print("Model loaded successfully!")

## 5. Helper Functions for Visualization

In [ ]:
def show_mask(mask, ax, random_color=False, alpha=0.6):
    """Display a segmentation mask on the given axes."""
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([alpha])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, alpha])
    
    # Handle tensor masks
    if hasattr(mask, 'cpu'):
        mask = mask.cpu().numpy()
    
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

def show_box(box, ax, label=None, color='green'):
    """Display a bounding box on the given axes."""
    # Handle tensor boxes
    if hasattr(box, 'cpu'):
        box = box.cpu().numpy()
    
    x0, y0, x1, y1 = box
    w, h = x1 - x0, y1 - y0
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor=color, facecolor=(0, 0, 0, 0), lw=2))
    if label:
        ax.text(x0, y0 - 5, label, color=color, fontsize=10, fontweight='bold')

def visualize_results(image, masks, boxes, scores, prompt, save_path=None):
    """Visualize segmentation results."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Original image
    axes[0].imshow(image)
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    
    # Image with masks
    axes[1].imshow(image)
    for i, (mask, box, score) in enumerate(zip(masks, boxes, scores)):
        show_mask(mask, axes[1], random_color=True)
        score_val = score.item() if hasattr(score, 'item') else score
        label = f"{prompt}: {score_val:.2f}"
        show_box(box, axes[1], label=label)
    axes[1].set_title(f"SAM 3 Segmentation: '{prompt}'")
    axes[1].axis('off')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        print(f"Saved: {save_path}")
    plt.show()

## 6. Run SAM 3 Inference

In [ ]:
def segment_image(image_path, text_prompt, threshold=0.5, mask_threshold=0.5):
    """Run SAM 3 segmentation on an image with a text prompt."""
    # Load image
    image = Image.open(image_path).convert("RGB")
    print(f"Image size: {image.size}")
    
    # Process inputs
    inputs = processor(images=image, text=text_prompt, return_tensors="pt").to(device)
    
    # Run inference
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Post-process results
    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=threshold,
        mask_threshold=mask_threshold,
        target_sizes=[image.size[::-1]]  # (height, width)
    )[0]
    
    masks = results["masks"]
    boxes = results["boxes"]
    scores = results["scores"]
    
    print(f"Found {len(masks)} instances of '{text_prompt}'")
    
    return image, masks, boxes, scores

In [ ]:
# Create output directory
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

### Example 1: Segment a Dog

In [ ]:
image, masks, boxes, scores = segment_image("images/dog.jpg", "dog")
visualize_results(image, masks, boxes, scores, "dog", "outputs/dog_segmented.png")

### Example 2: Segment Buildings in a City

In [ ]:
image, masks, boxes, scores = segment_image("images/city.jpg", "building")
visualize_results(image, masks, boxes, scores, "building", "outputs/city_segmented.png")

### Example 3: Segment Food Items

In [ ]:
image, masks, boxes, scores = segment_image("images/food.jpg", "food")
visualize_results(image, masks, boxes, scores, "food", "outputs/food_segmented.png")

### Example 4: Segment a Car

In [ ]:
image, masks, boxes, scores = segment_image("images/car.jpg", "car")
visualize_results(image, masks, boxes, scores, "car", "outputs/car_segmented.png")

### Example 5: Segment People

In [ ]:
image, masks, boxes, scores = segment_image("images/people.jpg", "person")
visualize_results(image, masks, boxes, scores, "person", "outputs/people_segmented.png")

## 7. Try Different Prompts on the Same Image

In [ ]:
# Try multiple prompts on the city image
prompts = ["window", "sky", "street", "car"]

fig, axes = plt.subplots(2, 2, figsize=(16, 16))
axes = axes.flatten()

for ax, prompt in zip(axes, prompts):
    image, masks, boxes, scores = segment_image("images/city.jpg", prompt)
    ax.imshow(image)
    for mask, box, score in zip(masks, boxes, scores):
        show_mask(mask, ax, random_color=True)
        show_box(box, ax, label=f"{score:.2f}")
    ax.set_title(f"Prompt: '{prompt}' ({len(masks)} found)")
    ax.axis('off')

plt.tight_layout()
plt.savefig("outputs/city_multi_prompt.png", bbox_inches='tight', dpi=150)
plt.show()

## 8. Batch Processing

In [ ]:
def batch_segment(image_paths, prompt):
    """Process multiple images with the same prompt."""
    results = []
    for path in image_paths:
        print(f"Processing: {path}")
        image, masks, boxes, scores = segment_image(path, prompt)
        results.append({
            "path": path,
            "image": image,
            "masks": masks,
            "boxes": boxes,
            "scores": scores,
            "count": len(masks)
        })
    return results

# Example: Find all "vehicles" across images
all_images = list(images_dir.glob("*.jpg"))
vehicle_results = batch_segment(all_images, "vehicle")

print("\nSummary:")
for r in vehicle_results:
    print(f"  {r['path'].name}: {r['count']} vehicles found")

## 9. Interactive Segmentation with Custom Image

In [ ]:
# You can add your own images to the 'images' folder and segment them
# Example:
# custom_image_path = "images/your_image.jpg"
# custom_prompt = "your object"
# image, masks, boxes, scores = segment_image(custom_image_path, custom_prompt)
# visualize_results(image, masks, boxes, scores, custom_prompt)

## Resources

- [SAM 3 GitHub Repository](https://github.com/facebookresearch/sam3)
- [SAM 3 Hugging Face](https://huggingface.co/facebook/sam3)
- [Meta AI Blog Post](https://ai.meta.com/blog/segment-anything-model-3/)
- [SAM 3 Demo](https://ai.meta.com/sam3/)